**Обучение LSTM-автоэнкодера и кластеризация режимов работы**

**Этапы:**
1. Формирование скользящих окон (ЭЦН: 60×9, ШГН: 120×9)
2. Обучение LSTM-автоэнкодера (encoder → latent space → decoder)
3. Построение латентного пространства (извлечение скрытых признаков)
4. Кластеризация латентных векторов (HDBSCAN/k-means)
5. Извлечение типовых профилей (декодирование центроидов кластеров)

**Вход:**
- data/processed/ecn_train.csv, ecn_val.csv, ecn_test.csv, scaler_ecn.pkl
- data/processed/shgn_train.csv, shgn_val.csv, shgn_test.csv, scaler_shgn.pkl

**Важно:** Данные ШГН уже объединены (скважины 134+135) и нормализованы одним scaler'ом.

**Выход:**
- models/ecn/ (encoder.pth, decoder.pth, clusters.pkl, profiles.pkl, metadata.json)
- models/shgn/ (encoder.pth, decoder.pth, clusters.pkl, profiles.pkl, metadata.json)

In [8]:
import pickle
from pathlib import Path

import pandas as pd

## Этап 1: Загрузка данных из /data/processed

In [9]:
DATA_DIR = Path("../data/processed")
FEATURE_COLUMNS = [
    "us_center",
    "us_periph",
    "gas_center",
    "gas_periph",
    "temp",
    "water_center",
    "water_periph",
    "gas_integral",
    "water_integral",
]

In [12]:
df_ecn_train = pd.read_csv(DATA_DIR / "ecn_train.csv", parse_dates=["timestamp"])
df_ecn_val = pd.read_csv(DATA_DIR / "ecn_val.csv", parse_dates=["timestamp"])
df_ecn_test = pd.read_csv(DATA_DIR / "ecn_test.csv", parse_dates=["timestamp"])

df_shgn_train = pd.read_csv(DATA_DIR / "shgn_train.csv", parse_dates=["timestamp"])
df_shgn_val = pd.read_csv(DATA_DIR / "shgn_val.csv", parse_dates=["timestamp"])
df_shgn_test = pd.read_csv(DATA_DIR / "shgn_test.csv", parse_dates=["timestamp"])

In [11]:
with open(DATA_DIR / "scaler_ecn.pkl", "rb") as f:
    scaler_ecn = pickle.load(f)

with open(DATA_DIR / "scaler_shgn.pkl", "rb") as f:
    scaler_shgn = pickle.load(f)

In [13]:
print("\nЭЦН:")
print(f"  Train: {len(df_ecn_train):,} точек")
print(f"  Val:   {len(df_ecn_val):,} точек")
print(f"  Test:  {len(df_ecn_test):,} точек")

print("\nШГН (скважины 134+135 объединённые):")
print(f"  Train: {len(df_shgn_train):,} точек")
print(f"  Val:   {len(df_shgn_val):,} точек")
print(f"  Test:  {len(df_shgn_test):,} точек")

print(f"\nПризнаков: {len(FEATURE_COLUMNS)}")
print(f"Колонки: {list(df_ecn_train.columns)}")


ЭЦН:
  Train: 40,770 точек
  Val:   5,091 точек
  Test:  5,096 точек

ШГН (скважины 134+135 объединённые):
  Train: 103,823 точек
  Val:   12,964 точек
  Test:  12,977 точек

Признаков: 9
Колонки: ['us_center', 'us_periph', 'gas_center', 'gas_periph', 'temp', 'water_center', 'water_periph', 'outlet_num', 'gas_integral', 'water_integral', 'timestamp', 'well']


In [14]:
print("\nПроверка нормализации:")
print(
    f"ЭЦН train - mean: {df_ecn_train[FEATURE_COLUMNS].mean().mean():.3f}, std: {df_ecn_train[FEATURE_COLUMNS].std().mean():.3f}"
)
print(
    f"ШГН train - mean: {df_shgn_train[FEATURE_COLUMNS].mean().mean():.3f}, std: {df_shgn_train[FEATURE_COLUMNS].std().mean():.3f}"
)

print("\nПропуски:")
print(f"ЭЦН: {df_ecn_train[FEATURE_COLUMNS].isnull().sum().sum()}")
print(f"ШГН: {df_shgn_train[FEATURE_COLUMNS].isnull().sum().sum()}")



Проверка нормализации:
ЭЦН train - mean: -0.034, std: 1.034
ШГН train - mean: -0.001, std: 1.049

Пропуски:
ЭЦН: 0
ШГН: 0


### Этап 2 — Формирование скользящих окон